<a href="https://colab.research.google.com/github/MatteoBaraldi/Machine-Learning-for-Bioengineering/blob/main/MOD-2/02_dimensionality_reduction/exercise_dimensionality_reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice: Dimensionality reduction algorithms

## Learning objectives
* apply PCA to a reduce the dimensionality of a high-dimensional dataset
* Select the optimal number of principal components
* repeat the exercise with IsoMap/tSNA/UMAP, and analyze how the projection depends on the model parameters

## 0. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.manifold import Isomap
from sklearn.manifold import TSNE

np.random.seed(42)

## 1. Import and refine data

The array X includes the expression value of 2000 genes for 2700 cells obtained by single-cell RNAseq. The kind of the cell, as defined by known genetic markers, is reported in the array y.

In [ ]:
#from google.colab import files
#uploaded = files.upload()
#for fn in uploaded.keys():
  #print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

In [ ]:
from google.colab import files
uploaded = files.upload() # Qui selezioni il file dal tuo PC

import numpy as np
data = np.load("data_pbmc.npz")

Remove from X and y all the cells that are labelled as 'Unknown'.
* How many cells are left ?
* Create an array, y_int, with the cell types converted into integer values, and an array, labels, with the list of cell types (Suggestion: use the function np.unique)

In [ ]:
# Cell 3: Remove 'Unknown' cells and create integer labels
inds = y != 'Unknown'
X = X[inds, :]
y = y[inds]

labels, y_int = np.unique(y, return_inverse=True)
print('Number of samples:', X.shape[0])
print('Cell types:', labels)

## 2. Principal Component Analysis

* Project the data onto the first 50 principal components
* Plot the cumulative variance explained by the PCs
* Plot the data over the first 2 PCs using different markers (or colors) for the various cell kinds
* Compute the silhoutte score using the first 2 PCs and the labels in y_int. How do you interpret the silhoutte score in this case ?

In [ ]:
# Cell 4: PCA
n_pc = 50
pca = PCA(n_components=n_pc)
pca.fit(X)
Xpca = pca.transform(X)

# Cumulative explained variance
f = plt.figure()
ax = f.add_subplot(1, 1, 1)
ax.plot(np.arange(n_pc), np.cumsum(pca.explained_variance_ratio_), '-ok')
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative explained variance')
ax.set_title('Explained variance vs number of PCs')
plt.tight_layout()
plt.show()

# Plot first 2 PCs
f = plt.figure()
ax = f.add_subplot(1, 1, 1)
for i_label, label in enumerate(labels):
    inds = y_int == i_label
    ax.plot(Xpca[inds, 0], Xpca[inds, 1], '+', label=label)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('PCA projection')
plt.legend()
plt.tight_layout()
plt.show()

# Silhouette score with first 2 PCs
sc = silhouette_score(Xpca[:, :2], y_int)
print(f'Silhouette score (2 PCs): {sc:.3f}')
print('Interpretation: values close to 1 indicate well-separated clusters.')
print('A moderate score suggests partial overlap between cell types in 2D PCA space.')

## 3. Dimensionality reduction with IsoMap

* Use the IsoMap method to project the data over a 2-dimensional space starting from the original data X
* Change the value of the number of neighbors in the range from 5 to 20, and select the value that optimize the silhoutte score. Can you get a decent separation of the different cell types ?
* Repeat the previous points using as input the projection of the data over the first 50 principal components
* Select the number of neighbors that provides the highest silhoutte score
* Plot the data in the projected space using different colors for the various cell types

In [ ]:
# Cell 5: IsoMap starting from original X
ns_neighbors = np.arange(5, 21, 2).astype(int)
scores_X = np.zeros(len(ns_neighbors))

for i, n_neighbors in enumerate(ns_neighbors):
    iso = Isomap(n_components=2, n_neighbors=n_neighbors)
    Xiso = iso.fit_transform(X)
    scores_X[i] = silhouette_score(Xiso, y_int)
    print(f'IsoMap (from X) | n_neighbors={n_neighbors}: silhouette={scores_X[i]:.3f}')

print(f'\nBest n_neighbors (from X): {ns_neighbors[np.argmax(scores_X)]}')
print('Can you get a decent separation? Likely limited due to high dimensionality.')

## 4. Dimensionality reduction with tSNE

* Use the TSNE method to project the data over a 2-dimensional space starting from the projection of the data over the first 50 principal components
* Test different values of perplexity and observe how the projection changes

In [ ]:
# Cell 7: tSNE
perplexity_values = [5, 10, 30, 50]

for perplexity in perplexity_values:
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    Xtsne = tsne.fit_transform(Xpca)

    f = plt.figure()
    ax = f.add_subplot(1, 1, 1)
    for i_label, label in enumerate(labels):
        inds = y_int == i_label
        ax.plot(Xtsne[inds, 0], Xtsne[inds, 1], '+', label=label)
    ax.set_xlabel('tSNE dim 1')
    ax.set_ylabel('tSNE dim 2')
    ax.set_title(f'tSNE (perplexity={perplexity})')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 4. Dimensionality reduction with UMAP

* Create a conda environment by cloning the base environment (conda create --name NAME_NEW_ENV --clone base)
* Activate the new environment (conda activate NAME_NEW_ENV)
* Install the package umap-learn from the channel conda-forge (conda install -c conda-forge umap-learn
* Import the umap library
* Use the umap method to project the data over a 2-dimensional space starting from the projection of the data over the first 50 principal components
* Test different values of the number of neighbors and observe how the projection changes

In [ ]:
# Cell 8: UMAP
# Install: pip install umap-learn
# or: conda install -c conda-forge umap-learn
import umap

n_neighbors_values = [5, 15, 30, 50]
min_dist = 0.1

for n_neighbors in n_neighbors_values:
    uma = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=42,
        n_jobs=1
    )
    Xuma = uma.fit_transform(Xpca)

    f = plt.figure()
    ax = f.add_subplot(1, 1, 1)
    for i_label, label in enumerate(labels):
        inds = y_int == i_label
        ax.plot(Xuma[inds, 0], Xuma[inds, 1], '+', label=label)
    ax.set_xlabel('UMAP dim 1')
    ax.set_ylabel('UMAP dim 2')
    ax.set_title(f'UMAP (n_neighbors={n_neighbors})')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Save best UMAP embedding for clustering
uma_best = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
    n_jobs=1
)
Xuma_best = uma_best.fit_transform(Xpca)

## 5. Clustering in the low dimensional space

* Clusterize the data in the low-dimensional space obtained by UMAP with DBSCAN. Optimize the parameters of DBSCAN by looking at the clusterized data in the low-dimensional space
* Use the function imshow to plot an heatmap of gene expression values. Restrict the scale of the heatmap in the range from -0.1 to 0.1
* Order the cells based on the DBSCAN clustering results

In [ ]:
# Cell 9: Clustering in the UMAP space with DBSCAN
from sklearn.cluster import DBSCAN

# Tune eps and min_samples by inspecting the scatter plot
model = DBSCAN(eps=0.5, min_samples=5).fit(Xuma_best)
n_clusters = np.max(model.labels_) + 1

f = plt.figure()
ax = f.add_subplot(1, 1, 1)
for i_label in range(n_clusters):
    inds = model.labels_ == i_label
    ax.plot(Xuma_best[inds, 0], Xuma_best[inds, 1], '+', label=str(i_label))
inds_noise = model.labels_ == -1
ax.plot(Xuma_best[inds_noise, 0], Xuma_best[inds_noise, 1], 'xk', label='noise')
ax.set_title('DBSCAN clustering on UMAP embedding')
ax.set_xlabel('UMAP dim 1')
ax.set_ylabel('UMAP dim 2')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Number of clusters: {n_clusters}')
print(f'Noise points: {np.sum(inds_noise)}')

# Heatmap ordered by DBSCAN cluster
sort_inds = np.argsort(model.labels_)
plt.figure(figsize=(12, 5))
plt.imshow(X[sort_inds], aspect='auto', cmap='viridis', vmin=-0.1, vmax=0.1)
plt.colorbar(label='Expression')
plt.xlabel('Gene index')
plt.ylabel('Cell (sorted by cluster)')
plt.title('Gene expression heatmap (sorted by DBSCAN cluster)')
plt.tight_layout()
plt.show()